# Hyper-MML + Soft Computing (Fuzzy Logic, Rough Sets, PSO)

This notebook runs the soft-computing extensions added to Hyper-MML (EEG + audio + video emotion recognition):

| # | Technique | Flag |
|---|---|---|
| 1 | Fuzzy-rough feature selection (QuickReduct) | `--rs_select quickreduct` |
| 2 | Fuzzy-rough reduct via Genetic Algorithm | `--rs_select ga` |
| 3 | Rough boundary-region sample weighting | `--rs_weight` |
| 4 | Fuzzy TSK modality-reliability gate | `--fuzzy_gate` |
| 5 | Particle Swarm Optimisation of hyper-parameters | `tune_pso.py` |

**Run the cells top to bottom.** Start with `DATA = "synthetic"` to check everything works,
then set `DATA = "real"` once you have the preprocessed EAV `.pkl` files.

## 1. Setup

In [ ]:
import os, sys, json, subprocess, tempfile, time
from pathlib import Path

# --- locate the repo (folder containing train.py) ---
REPO = Path.cwd()
if not (REPO / "train.py").exists():
    REPO = next(p for p in [Path.cwd().parent, Path.cwd() / "Hyper-MML"] if (p / "train.py").exists())
sys.path.insert(0, str(REPO))
print("repo:", REPO)

# --- dependency check (on Colab / fresh env run the pip line below once) ---
# !pip install torch torch_geometric numpy pandas scikit-learn scipy matplotlib
import importlib
missing = [m for m in ["torch", "torch_geometric", "numpy", "pandas", "sklearn", "matplotlib"] if importlib.util.find_spec(m) is None]
assert not missing, f"missing packages: {missing}  ->  pip install torch torch_geometric numpy pandas scikit-learn matplotlib"

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)

## 2. Configuration

* `DATA = "synthetic"` generates fake EAV-format features (pipeline check only, **numbers are not research results**).
* `DATA = "real"` expects the preprocessed files from the README at `REAL_ROOT/EAV/features/{Audio_openSMILE,Vision_manet,EEG_NESTA,lab}/subject_XX_*.pkl`.

In [ ]:
DATA      = "synthetic"                 # "synthetic" or "real"
REAL_ROOT = "/path/to/dataset"          # folder that contains EAV/features/...   (only used when DATA == "real")
SUBJECT   = 1
EPOCHS    = 30
VALID_FRAC = 0.2                        # held-out validation split -> honest model selection (never uses test set)

## 3. Data

In [ ]:
import make_synthetic

if DATA == "synthetic":
    DATA_ROOT = Path(tempfile.gettempdir()) / "hmml_synth"
    INFO_IDX = make_synthetic.make(str(DATA_ROOT / "EAV"), SUBJECT)      # indices of truly informative features
    print("generated synthetic data in", DATA_ROOT)
else:
    DATA_ROOT = Path(REAL_ROOT)
    INFO_IDX = None
    need = [DATA_ROOT / "EAV/features" / d / f"subject_{SUBJECT:02d}_{s}.pkl"
            for d, s in [("Audio_openSMILE", "aud"), ("Vision_manet", "vis"), ("EEG_NESTA", "eeg"), ("lab", "lab")]]
    for p in need:
        assert p.exists(), f"missing {p}"
    print("real data found")

os.environ["HYPERMML_ROOT"] = str(DATA_ROOT)       # read by datasets.py (must be set before importing it)
from datasets import EAVDataset
trainset, testset = EAVDataset(train=True, subject=SUBJECT), EAVDataset(train=False, subject=SUBJECT)
y_train = trainset.labels(True)
print("train samples:", len(trainset), "| test samples:", len(testset), "| classes:", np.bincount(y_train))
for m in ("eeg", "aud", "vis"):
    print(f"  {m}: {trainset.mod_array(m, True).shape[1]} features")

## 4. Rough sets: fuzzy-rough reduct search

For each modality we search a *reduct* – a small feature subset that keeps the emotion classes discernible.
`gamma` is the fuzzy-rough **dependency degree** (how much of the data lies in the positive region).
QuickReduct is greedy; the GA searches subsets directly (fitness = gamma, penalised by subset size).

In [ ]:
from softcomputing import FuzzyRoughSelector, rough_sample_weights

mod = "aud"
X = trainset.mod_array(mod, True)
t = time.time(); qr = FuzzyRoughSelector(sigma=3.0, max_features=48).fit(X, y_train, method="quickreduct"); t_qr = time.time() - t
t = time.time(); ga = FuzzyRoughSelector(sigma=3.0, max_features=48).fit(X, y_train, method="ga");         t_ga = time.time() - t
print(f"QuickReduct: {len(qr.selected_)} features in {t_qr:.1f}s | GA: {len(ga.selected_)} features in {t_ga:.1f}s")
if INFO_IDX is not None:
    for name, s in [("QuickReduct", qr), ("GA", ga)]:
        print(f"{name}: recovered {len(set(s.selected_) & set(INFO_IDX[mod]))}/{len(INFO_IDX[mod])} planted informative features")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(range(1, len(qr.history_) + 1), qr.history_, marker="o", ms=3)
ax[0].axhline(qr.gamma_full_, ls="--", c="gray", label="gamma (candidate pool)")
ax[0].set(title="QuickReduct: dependency degree", xlabel="# features selected", ylabel="gamma"); ax[0].legend()
ax[1].plot(ga.history_); ax[1].set(title="GA: best fitness per generation", xlabel="generation", ylabel="fitness")
plt.tight_layout(); plt.show()

In [ ]:
# Positive-region membership -> boundary-region (ambiguous) samples
mu = qr.positive_region_membership(X, y_train)
w = rough_sample_weights(mu, floor=0.3)
plt.figure(figsize=(5, 3)); plt.hist(mu, bins=30); plt.title("Positive-region membership of training samples")
plt.xlabel("membership (low = boundary / ambiguous)"); plt.show()
print(f"{(w < 0.65).sum()} of {len(w)} training samples are down-weighted to < 0.65")

## 5. Ablation: baseline vs. soft-computing variants

Each configuration trains Hyper-MML via `train.py` and reports **test F1 at the best-validation epoch**
(the original script's "max test F1 over epochs" is also shown, but it selects the epoch on the test set).

In [ ]:
CONFIGS = {
    "1 baseline (Hyper-MML)":            [],
    "2 + fuzzy-rough QuickReduct":       ["--rs_select", "quickreduct"],
    "3 + fuzzy-rough GA":                ["--rs_select", "ga"],
    "4 + fuzzy TSK gate":                ["--fuzzy_gate"],
    "5 + QuickReduct + boundary weights":["--rs_select", "quickreduct", "--rs_weight"],
    "6 ALL (GA + weights + gate)":       ["--rs_select", "ga", "--rs_weight", "--fuzzy_gate"],
}

def run_train(flags, epochs=EPOCHS, subject=SUBJECT):
    cmd = [sys.executable, str(REPO / "train.py"), "--subject", str(subject), "--epochs", str(epochs),
           "--valid_frac", str(VALID_FRAC)] + (["--no-cuda"] if DEVICE == "cpu" else []) + flags
    with tempfile.TemporaryDirectory() as wd:            # train.py writes record_*.pk to cwd
        p = subprocess.run(cmd, cwd=wd, capture_output=True, text=True,
                           env=dict(os.environ, PYTHONPATH=str(REPO), HYPERMML_ROOT=str(DATA_ROOT)))
    res = next((json.loads(l[7:]) for l in p.stdout.splitlines() if l.startswith("RESULT ")), None)
    if res is None:
        print(p.stderr[-2000:])
    return res, p.stdout

results, logs = {}, {}
for name, flags in CONFIGS.items():
    t = time.time(); results[name], logs[name] = run_train(flags)
    r = results[name]; print(f"{name:<38} valid F1 {r['best_valid_f1']:.2f} | test F1@bestVal {r['test_f1_at_best_valid']:.2f}  ({time.time()-t:.0f}s)")

In [ ]:
df = pd.DataFrame(results).T[["best_valid_f1", "test_f1_at_best_valid", "test_acc_at_best_valid", "test_f1_max_over_epochs", "n_features"]]
df.columns = ["valid F1", "test F1 @ best valid", "test Acc @ best valid", "test F1 (max over epochs, optimistic)", "#features (eeg,aud,vis)"]
display(df)
ax = df["test F1 @ best valid"].astype(float).plot.barh(figsize=(7, 3.2), color="#4c72b0")
ax.set_xlabel("test weighted-F1 (%)"); ax.invert_yaxis(); plt.tight_layout(); plt.show()

## 6. Fuzzy gate: learned modality trust and rule base

The gate turns *certainty* and *agreement* of each modality into fuzzy sets (LOW / MEDIUM / HIGH) and applies 9 TSK rules.
Mean trust per modality should drift away from 1.0 if some modality is consistently more reliable.

In [ ]:
_, out = run_train(["--fuzzy_gate"])
lines = out.splitlines()
start = next(i for i, l in enumerate(lines) if l.startswith("Fuzzy gate"))
print("\n".join(lines[start:start + 25]))

## 7. PSO hyper-parameter tuning

Particles are configurations (lr, l2, dropout, #hypergraph layers, rough-set sigma, reduct size, aux-loss weight).
Fitness = **validation** F1 (test set never used). Defaults are small; increase `PARTICLES` / `ITERS` for a real search.

In [ ]:
from softcomputing import PSO
from tune_pso import SPACE, decode, fitness_of

PARTICLES, ITERS, PSO_EPOCHS = 4, 2, 15
base = ["--rs_select", "quickreduct", "--rs_weight", "--fuzzy_gate"]

def evaluate_batch(xs):
    out = []
    for x in xs:
        cfg = decode(x)
        res, _ = run_train(base + sum([["--" + k, str(v)] for k, v in cfg.items()], []), epochs=PSO_EPOCHS)
        f = fitness_of(res); out.append(f); print("   ", cfg, "-> valid F1", f)
    return out

best_x, best_f, hist = PSO([b for _, b in SPACE], n_particles=PARTICLES, iters=ITERS, seed=0).run(evaluate_batch)
best_cfg = decode(best_x)
print("\nbest config:", best_cfg, "| valid F1:", best_f)
plt.plot(hist, marker="o"); plt.xlabel("PSO iteration"); plt.ylabel("best validation F1"); plt.show()

In [ ]:
final, _ = run_train(base + sum([["--" + k, str(v)] for k, v in best_cfg.items()], []), epochs=PSO_EPOCHS)
print("tuned config -> test F1 @ best valid:", final["test_f1_at_best_valid"], "| test acc:", final["test_acc_at_best_valid"])

## 8. (Optional) Run the same experiments on Modal GPUs

Needs the `modal` CLI to be logged in (`modal setup`). For real data first upload it once:
`modal volume put hypermml-data <local EAV folder> /EAV`.

In [ ]:
# Uncomment to run. Synthetic data:
# !cd {REPO} && modal run modal_app.py::smoke --epochs 30
# Real data, one subject:
# !cd {REPO} && modal run modal_app.py::smoke --data real --subject 1
# PSO on Modal (swarm evaluated in parallel):
# !cd {REPO} && modal run modal_app.py::pso --data real --subject 1 --particles 6 --iters 4

## Notes for the report
* Selection of epochs / hyper-parameters uses the validation split only; report **test F1 @ best valid**.
* Feature selection and boundary weights are fitted on the training split only (no test leakage).
* Run several subjects and seeds before drawing conclusions; single-run differences of a few points are within noise.
* The EAV paper uses subject-dependent models, so average over subjects 1-42 for the final table.